# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 1 — Probability models and checking data sources

## What you will do

In this session you will build a small probability model, calculate with it,
and use simulation to see whether its predictions are plausible. You will also
check where a data file came from and what its limitations are before using it.

Before you start: Chapter 1, Sections 1.1–1.3. By the end, you will have a
short probability calculation and a source summary for the bundled text file.


In [ ]:
from collections import Counter
from fractions import Fraction
from hashlib import sha256
import re

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(2026)

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )


## 1. Start with a simple probability model

Roll two independent fair six-sided dice. The outcome space is the 36 ordered pairs
$\Omega=\{(i,j):i,j\in\{1,\ldots,6\}\}$, every subset is an event, and every outcome has probability $1/36$.

Let $A$ be the event that the sum is at least 10, and let $B$ be the event that the first die is even. The pair $(4,6)$ is an outcome, $\{(4,6)\}$ is a singleton event, and the sum $10$ is a numerical value of the random variable $S(i,j)=i+j$.


In [ ]:
omega = {(i, j) for i in range(1, 7) for j in range(1, 7)}
A = {(i, j) for i, j in omega if i + j >= 10}
B = {(i, j) for i, j in omega if i % 2 == 0}

P_A = len(A) / len(omega)
P_B = len(B) / len(omega)
P_A_and_B = len(A & B) / len(omega)
P_A_given_B = P_A_and_B / P_B

print(f"P(A) = {P_A:.3f}")
print(f"P(B) = {P_B:.3f}")
print(f"P(A intersection B) = {P_A_and_B:.3f}")
print(f"P(A given B) = {P_A_given_B:.3f}")

assert len(omega) == 36
assert np.isclose(P_B, 0.5)
assert P_B > 0


The conditioning event must have positive probability. Independence is an assumption in the model, not a consequence of saying that two rolls were performed.


## 2. Checking a model with relative frequency

The probability $P(A)$ was defined by the finite law above. The running relative frequency below should approach it under independent repeated sampling; the law of large numbers explains that later in the course.


In [ ]:
n_trials = 5_000
rolls = rng.integers(1, 7, size=(n_trials, 2))
hits = (rolls.sum(axis=1) >= 10).astype(float)
running_frequency = np.cumsum(hits) / np.arange(1, n_trials + 1)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(running_frequency, label="running relative frequency")
ax.axhline(P_A, color="black", linestyle="--", label="exact P(A)")
ax.set(xlabel="number of independent trials", ylabel="relative frequency")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

print(f"Final relative frequency: {running_frequency[-1]:.3f}")


## 3. How sampling design can cause dependence

Two draws with replacement from three labelled cards are independent when each draw is uniform. Two draws without replacement are dependent. Repetition alone does not decide independence.


In [ ]:
cards = np.array(["A", "B", "C"])
n_rep = 20_000

with_replacement = rng.choice(cards, size=(n_rep, 2), replace=True)
same_with = np.mean(with_replacement[:, 0] == with_replacement[:, 1])

first = rng.choice(cards, size=n_rep, replace=True)
second_without = np.array(
    [rng.choice(cards[cards != value]) for value in first]
)
same_without = np.mean(first == second_without)

print(f"P(equal), with replacement:    about {same_with:.3f} (exactly 1/3)")
print(f"P(equal), without replacement: about {same_without:.3f} (exactly 0)")


## 4. Checking the data before analysis

The local file data/pride_and_prejudice.txt contains its own Project Gutenberg header and licence text. A repository copy is convenient, but it does not remove the duty to record the source and check whether redistribution is permitted in the setting where the work will be published.

Source summary:

- source and version: Project Gutenberg eBook 1342; the bundled header reports an update on 10 March 2018;
- observational unit: one token produced by the explicit tokenisation rule below;
- intended use: a reproducible word-frequency illustration, not a claim about English in general;
- licence: the bundled Project Gutenberg terms govern reuse; jurisdiction and current terms must be checked before redistribution;
- privacy: this is a published literary text, not a collection of student or platform-user records;
- integrity: record a checksum so that later analyses can identify the exact local copy.


In [ ]:
text_path = course_data("pride_and_prejudice.txt")
raw_bytes = text_path.read_bytes()
text = raw_bytes.decode("utf-8-sig")
words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", text.lower())
counts = Counter(words)

print("SHA-256:", sha256(raw_bytes).hexdigest())
print("Number of parsed tokens:", len(words))
print("Ten most frequent tokens:", counts.most_common(10))


In [ ]:
top = counts.most_common(12)
labels, values = zip(*top)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(labels, values)
ax.set(xlabel="token", ylabel="count", title="Most frequent parsed tokens")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


## 5. Urn model

An urn contains two red balls and six black balls. Label the red balls 1 and 2 and the black balls 3 through 8. We draw three balls sequentially, uniformly, and without replacement.

An outcome must record the order of the draws. Thus $(1,3,5)$ and $(3,1,5)$ are different outcomes. The outcome space is

$$\Omega_{\mathrm{urn}}=\{(i,j,k):i,j,k\in\{1,\ldots,8\}\text{ are distinct}\}.$$

There are $8\cdot7\cdot6=336$ outcomes. Each has probability $1/336$ because every available ball is equally likely at each draw. Define

- $A$: at least one of the three balls is red;
- $B$: exactly one of the three balls is red;
- $C$: the second ball is black.

Before running the next cell, decide whether $A$ and $B$ can be independent. Notice the set relation between them.

In [ ]:
# Ball labels make individual balls distinguishable even when their colours agree.
red = {1, 2}
black = {3, 4, 5, 6, 7, 8}

# Ordered triples preserve which ball was drawn first, second, and third.
urn_omega = {
    (first, second, third)
    for first in range(1, 9)
    for second in range(1, 9)
    for third in range(1, 9)
    if len({first, second, third}) == 3
}

# Count red labels in an outcome; this makes all three event definitions easy to read.
number_of_red = lambda outcome: len(red & set(outcome))
A_urn = {outcome for outcome in urn_omega if number_of_red(outcome) >= 1}
B_urn = {outcome for outcome in urn_omega if number_of_red(outcome) == 1}
C_urn = {outcome for outcome in urn_omega if outcome[1] in black}

# Uniform outcomes let us compute probabilities by counting. Fraction keeps them exact.
def urn_probability(event):
    return Fraction(len(event), len(urn_omega))

P_A_urn = urn_probability(A_urn)
P_B_urn = urn_probability(B_urn)
P_A_and_B_urn = urn_probability(A_urn & B_urn)
P_B_given_A_urn = P_A_and_B_urn / P_A_urn
P_C_urn = urn_probability(C_urn)

print(f"Number of ordered outcomes: {len(urn_omega)}")
print(f"P(A) = {P_A_urn} = {float(P_A_urn):.3f}")
print(f"P(B) = {P_B_urn} = {float(P_B_urn):.3f}")
print(f"P(A intersection B) = {P_A_and_B_urn}")
print(f"P(B given A) = {P_B_given_A_urn} = {float(P_B_given_A_urn):.3f}")
print(f"P(C) = {P_C_urn} = {float(P_C_urn):.3f}")
print(f"Are A and B independent? {P_A_and_B_urn == P_A_urn * P_B_urn}")

assert len(urn_omega) == 8 * 7 * 6
assert B_urn <= A_urn
assert P_C_urn == Fraction(6, 8)

The counts agree with direct calculations. For example,

$$P(A)=1-P(\text{all three are black})
=1-\frac{6}{8}\frac{5}{7}\frac{4}{6}
=\frac{9}{14}.$$

For $B$, choose the red ball, choose two of the six black balls, and arrange the three chosen balls in draw order. This gives

$$P(B)=\frac{2\binom{6}{2}3!}{8\cdot7\cdot6}=\frac{15}{28}.$$

Since $B\subseteq A$, we have $A\cap B=B$. Therefore $P(A\cap B)=15/28$, whereas $P(A)P(B)=135/392$. The events are not independent. Equivalently, $P(B\mid A)=5/6\ne P(B)$. Finally, every draw position is marginally uniform over the eight balls, so $P(C)=6/8=3/4$.

### Why an unordered outcome space loses information

Suppose instead that we store each selection as an increasing triple. This smaller space records which three balls were selected, but it discards their draw order. It is still adequate for $A$ and $B$ because these events depend only on the colours selected. Each increasing triple represents exactly $3!=6$ ordered outcomes, so uniform probability on the increasing triples gives the same probabilities for such order-insensitive events.

It cannot represent $C$. In an increasing triple, index 1 is the ball with the middle label, not the ball drawn second. The calculation called `middle_label_is_black` below is therefore not a calculation of $P(C)$.

In [ ]:
# Increasing triples represent unordered selections of three balls.
unordered_omega = {
    (smallest, middle, largest)
    for smallest in range(1, 9)
    for middle in range(smallest + 1, 9)
    for largest in range(middle + 1, 9)
}

A_unordered = {outcome for outcome in unordered_omega if number_of_red(outcome) >= 1}
B_unordered = {outcome for outcome in unordered_omega if number_of_red(outcome) == 1}

# This is only an attempted translation of C. The middle entry is not a draw position.
middle_label_is_black = {
    outcome for outcome in unordered_omega if outcome[1] in black
}

def unordered_probability(event):
    return Fraction(len(event), len(unordered_omega))

print(f"Number of unordered selections: {len(unordered_omega)}")
print(f"P(A) from unordered selections = {unordered_probability(A_unordered)}")
print(f"P(B) from unordered selections = {unordered_probability(B_unordered)}")
print(
    "P(middle label is black), which is not P(C), =",
    unordered_probability(middle_label_is_black),
)

assert len(unordered_omega) == 56  # This is C(8, 3).
assert unordered_probability(A_unordered) == P_A_urn
assert unordered_probability(B_unordered) == P_B_urn
assert unordered_probability(middle_label_is_black) != P_C_urn

## Recap

Before you finish, make sure you can:

1. Recompute the exact dice probability of $A$ by listing the possible sums.
2. Explain why a simulation provides evidence about a probability model but does not define the model probability.
3. Explain how sampling with or without replacement changes dependence.
4. State why the urn model uses ordered triples, and explain why unordered triples work for $A$ and $B$ but not for the position-dependent event $C$.
5. Check independence either by comparing $P(A\cap B)$ with $P(A)P(B)$ or, when the conditioning probability is positive, by comparing $P(B\mid A)$ with $P(B)$.
6. Add one limitation to the source summary and state why the token counts do not estimate a universal English-language distribution.
7. Record the checksum with your analysis so that another person can identify the same source file.